For each NACE Class get the 100 chunks that scored highest across all the reports 

In [1]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
from test_base import *

In [2]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [3]:
overview_path = "data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"

In [4]:
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"

In [5]:
reports = glob.glob(raw_data_path + "*/*_short.csv")
reports = glob.glob(raw_data_path + "*/*_long.csv")
len(reports)

1556

In [6]:
#sample_ratio = 1

In [7]:
#max_elements_per_class = 1000000
#top_k_sentences = 200000

In [8]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
#filter_only_right_chunks = True

In [9]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
#with_null_classifiers = False

In [10]:
new_threshold_cos_sin = 0.5

In [11]:
nace_level_descriptions = 2
nace_level = 1
assert nace_level_descriptions >= nace_level

In [12]:
training_data_path = "data/training_data/"

In [13]:
#suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"
suffix = f"2nd_approach" + f"__nace_level_{nace_level}__cos_thres_{new_threshold_cos_sin}"

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path
os.makedirs(end_path, exist_ok=True)

In [14]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path)
df_overview.head()

,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
1,35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
2,80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
3,49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
4,73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [15]:
df_nace_codes_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

For each class c: those paragraphs p of reports in class c with cos-sim(p, c) > 0.5

In [16]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    df = pd.read_csv(report)
    
    report_name = os.path.basename(report).replace(".txt_long.csv", "") + ".pdf"
    report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
    report_code = get_all_level(report_code)[nace_level]

    scores = [c for c in df.columns if "Scores" in c]
    
    df["max_class_sim"] = [scores[i][7] for i in np.argmax(df[scores], 1)]
    df["Score"] = df[scores].max(1)
    
    df.loc[(df["max_class_sim"] == report_code) & (df["Score"] > new_threshold_cos_sin),"NACE_Code"] = report_code
    df["NACE_Code"] = df["NACE_Code"].fillna("NO_CLASS")

    result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])


  0%|                                                                                                                                                                                        | 0/1556 [00:00<?, ?it/s]/tmp/ipykernel_3698821/640071683.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1556/1556 [01:27<00:00, 17.81it/s]


In [17]:
stats = result.groupby("NACE_Code").count()["Sentences"]
stats

NACE_Code
A                74
B              1202
C               145
D              1482
E               920
F              2397
G               402
H              1088
I               254
J               207
K              2713
L               754
M               142
N                65
NO_CLASS    1653534
P               116
Q                85
R                31
S                 1
T                 3
Name: Sentences, dtype: int64

In [18]:
amount_no_class = stats[stats.index != "NO_CLASS"].max().item()
amount_no_class

2713

In [19]:
result_right = result[result["NACE_Code"] != "NO_CLASS"]
result_NO_CLASS = result.loc[result["NACE_Code"] == "NO_CLASS"].sample(n=amount_no_class)
result_final = pd.concat([result_right, result_NO_CLASS], axis=0)

In [20]:
stats = result.groupby("NACE_Code").agg({
    "Sentences": "count", 
    "Score": "mean"
})
stats

,Sentences,Score
NACE_Code,,
A,74,0.527524
B,1202,0.538372
C,145,0.537428
D,1482,0.540909
E,920,0.552916
F,2397,0.554142
G,402,0.533678
H,1088,0.544383
I,254,0.540151


In [21]:
stats.to_csv(end_path + "/stats.csv")

In [22]:
# recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

# df_recordings = pd.DataFrame(recordings)
# df_recordings = df_recordings.sort_values(by="Code")
# df_recordings.head()

#df_recordings.to_csv(end_path + "/statistics.csv")

In [23]:
full_df= result_final.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,text,Score,NACE_Code
45,the group has in place special standards for m...,0.770882,E
278,nature of business location type of service co...,0.526341,E
292,the groups intangible assets other than goodwi...,0.532391,E
350,the group is generally entitled to use all of ...,0.620223,E
386,i contract assets relating to service concessi...,0.553546,E
...,...,...,...
1280,poridgc. procedure for maintaining asbuilt doc...,0.415890,NO_CLASS
75,financing and retained earnings. financial ser...,0.610018,NO_CLASS
81,l the margin would be required to decrease by ...,0.187492,NO_CLASS
1053,during the meetings of the board of directors ...,0.390497,NO_CLASS


In [33]:
# Store each class for reading
n = 40
for nace_class in stats.index: 
    print(nace_class)
    temp = full_df[full_df["NACE_Code"] == nace_class].copy()
    temp["Evaluation"] = None
    temp["Notes"] = None
    if len(temp) >= n:
        temp = temp.sample(n=40)
    temp.to_csv(os.path.join(end_path, nace_class + ".csv"))

A
B
C
D
E
F
G
H
I
J
K
L
M
N
NO_CLASS
P
Q
R
S
T


In [25]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 8876, Test size: 2959, Validation size: 2959


In [26]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [27]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [28]:
end_path

'data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.5'

In [29]:
df_ = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.45/full_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'projects/nace_classification/nace_report_topic_analysis/data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.45/full_data.csv'

In [ ]:
df_.groupby

,text,Score,NACE_Code
0,revenue from the sale of electricity and envir...,0.468761,D
1,the following significant modifications to out...,0.464732,N
2,our operating results are affected by certain ...,0.450665,N
3,for more information see item . additional inf...,0.466704,N
4,in addition to the information set forth in th...,0.473458,N
...,...,...,...
41276,domestic connector factories have long been de...,0.300985,NO_CLASS
41277,for risks associated with the contractual arra...,0.420872,NO_CLASS
41278,by request other managers from outside group m...,0.416538,NO_CLASS
41279,operating segments are reported in a manner co...,0.433155,NO_CLASS
